# Treino LoRA com armazenamento persistente

Depende da sessão já preparada com modelo_lora, tokenizer, exemplos_treino_tokenizados, estado_inicial, MODEL_ID e MODEL_REVISION.

A primeira execução concluiu559 atualizações, mas seus pesos ficaram na VM temporária e não foram preservados. Seus relatórios permanecem em reports/training_epoch1*.json. Esta segunda execução mantém hiperparâmetros e salva checkpoints, adaptador final, registros e ZIP diretamente no Drive. Não sobrescreve a primeira execução nem permite iniciar sobre checkpoints existentes. Se houver interrupção, retomar um checkpoint em etapa específica.

Esta versão foi preparada para execução no Colab; seu resultado ainda não foi recebido. Checkpoints a cada100 atualizações, dois mais recentes; nenhuma avaliação de teste nesta célula.


In [ ]:
import gc
import json
import math
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    Trainer, TrainingArguments, DataCollatorForSeq2Seq, set_seed,
)
from peft import get_peft_model_state_dict

set_seed(42)
assert Path('/content/drive/MyDrive/Triunfo').is_dir(), 'Conecte o Drive primeiro.'
pasta_treino = Path('/content/drive/MyDrive/Triunfo/runs/lora_epoch1_recovery')
pasta_final = pasta_treino / 'adapter_final'
assert not pasta_final.exists(), 'Já há um adaptador final. Não repetir o treino.'
assert not list(pasta_treino.glob('checkpoint-*')), (
    'Há checkpoints anteriores. Precisamos retomar, em vez de começar novamente.'
)
assert len(exemplos_treino_tokenizados) == 4469
assert all(
    torch.equal(t.detach().cpu(), estado_inicial[n])
    for n, t in get_peft_model_state_dict(modelo_lora).items()
), 'Os adaptadores precisam estar no estado inicial preservado no piloto.'

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

colador = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=None, padding=True,
    pad_to_multiple_of=8, label_pad_token_id=-100,
    return_tensors='pt',
)
# Confere que o preenchimento preserva a supervisão existente.
amostra_lote = colador(exemplos_treino_tokenizados[:2])
for i, original in enumerate(exemplos_treino_tokenizados[:2]):
    n = len(original['labels'])
    assert amostra_lote['labels'][i, :n].tolist() == original['labels']
    assert (amostra_lote['labels'][i, n:] == -100).all().item()
del amostra_lote

argumentos_treino = TrainingArguments(
    output_dir=str(pasta_treino),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=28,
    lr_scheduler_type='linear',
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='adamw_torch',
    logging_steps=25,
    logging_nan_inf_filter=False,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    eval_strategy='no',
    report_to='none',
    seed=42,
    data_seed=42,
    dataloader_num_workers=0,
    push_to_hub=False,
)
pasta_treino.mkdir(parents=True, exist_ok=True)
(pasta_treino / 'training_arguments.json').write_text(
    argumentos_treino.to_json_string(), encoding='utf-8',
)

modelo_lora.config.use_cache = False
trainer = Trainer(
    model=modelo_lora,
    args=argumentos_treino,
    train_dataset=Dataset.from_list(exemplos_treino_tokenizados),
    data_collator=colador,
    processing_class=tokenizer,
)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print('Exemplos: 4469 | Épocas: 1 | Atualizações previstas:', math.ceil(4469 / 8))
resultado_treino = trainer.train()

trainer.save_model(str(pasta_final))
tokenizer.save_pretrained(str(pasta_final))
trainer.save_state()
trainer.save_metrics('train', resultado_treino.metrics)
modelo_lora.gradient_checkpointing_disable()
modelo_lora.config.use_cache = True
modelo_lora.eval()

resumo_treino = {
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'training_examples': 4469,
    'global_step': trainer.state.global_step,
    'epoch': trainer.state.epoch,
    'metrics': resultado_treino.metrics,
    'gpu_peak_allocated_gib': round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    'adapter_path': str(pasta_final),
}
(pasta_treino / 'training_epoch1.json').write_text(
    json.dumps(resumo_treino, ensure_ascii=False, indent=2), encoding='utf-8',
)
print(json.dumps(resumo_treino, ensure_ascii=False, indent=2))

# Backup persistente do resultado final e dos registros.
import hashlib
import zipfile
from google.colab import files

assert (pasta_final / 'adapter_config.json').is_file()
assert (pasta_final / 'adapter_model.safetensors').is_file()
arquivos_backup = sorted(
    [p for p in pasta_final.rglob('*') if p.is_file()]
    + list(pasta_treino.glob('*.json'))
)
manifesto_backup = {
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'global_step': trainer.state.global_step,
    'files': {
        p.relative_to(pasta_treino).as_posix():
        hashlib.sha256(p.read_bytes()).hexdigest()
        for p in arquivos_backup
    },
}
zip_final = pasta_treino / 'triunfo-lora-epoch1-backup.zip'
with zipfile.ZipFile(zip_final, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in arquivos_backup:
        z.write(p, p.relative_to(pasta_treino).as_posix())
    z.writestr('backup_manifest.json', json.dumps(manifesto_backup, indent=2))
with zipfile.ZipFile(zip_final) as z:
    for nome, esperado in manifesto_backup['files'].items():
        assert hashlib.sha256(z.read(nome)).hexdigest() == esperado
print('Backup conferido e salvo no Drive:', zip_final)
files.download(str(zip_final))
